# 🌱 Soil Carbon Agentic AI — Research-Grade Pipeline
### Stratified Sampling + Spatial Cross Validation + LangGraph

---
| Cell | Description |
|------|-------------|
| 1 | Install packages |
| 2 | Imports + path setup |
| 3 | GEE authentication |
| 4 | **Stratified data** (equal samples per SOC stratum) |
| 5 | Spectral indices |
| 6 | Land practice labels |
| 7 | **Spatial CV split** (geographic fold assignment) |
| 8 | **Train model with spatial CV** (honest accuracy) |
| 9 | **Stratum-wise accuracy** (per SOC level) |
| 10 | Visualizations |
| 11 | Interactive map |
| 12 | Agent pipeline |
| 13 | Save + report |
| 14 | Launch dashboard |

> **Key upgrade:** Random sampling → Stratified sampling + Spatial CV
> This is the research-grade approach required for paper submission.

## 📦 Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

packages = [
    'earthengine-api','geemap','geopandas','rasterio',
    'numpy','pandas','scikit-learn','matplotlib','seaborn',
    'folium','streamlit','streamlit-folium',
    'langchain','langchain-openai','langgraph==0.0.38',
    'requests','python-dotenv','shapely','plotly','Pillow','ipywidgets'
]

failed = []
print('Installing...')
for pkg in packages:
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'],
                       capture_output=True, text=True)
    status = '✅' if r.returncode==0 else '❌'
    if r.returncode != 0: failed.append(pkg)
    print(f'  {status} {pkg}')

print('\n🎉 Done!' if not failed else f'Failed: {failed}')
print('   Restart kernel if first install (Kernel → Restart)')

Installing...
  ✅ earthengine-api
  ✅ geemap
  ✅ geopandas
  ✅ rasterio
  ✅ numpy
  ✅ pandas
  ✅ scikit-learn
  ✅ matplotlib
  ✅ seaborn
  ✅ folium
  ✅ streamlit
  ✅ streamlit-folium
  ✅ langchain
  ✅ langchain-openai
  ✅ langgraph==0.0.38
  ✅ requests
  ✅ python-dotenv
  ✅ shapely
  ✅ plotly
  ✅ Pillow
  ✅ ipywidgets

🎉 Done!
   Restart kernel if first install (Kernel → Restart)


## 📚 Cell 2 — Imports & Project Path

In [2]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')

# ⚠️ Set your project root here
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
# If auto-detect fails, use manual path:
# PROJECT_ROOT = r'C:\Users\akiran\soil-carbon-agent'

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
os.makedirs('outputs', exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('dark_background')

# Verify all src files exist
files = ['src/ingestion.py','src/indices.py','src/soc_model.py','src/agent.py']
print('📁 File check:')
for f in files:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f'  {status}  {f}')
print(f'\n✅ Project root: {PROJECT_ROOT}')

📁 File check:
  ✅  src/ingestion.py
  ✅  src/indices.py
  ✅  src/soc_model.py
  ✅  src/agent.py

✅ Project root: C:\Users\akiran\soil-carbon-agent


## 🛰️ Cell 3 — GEE Authentication
> Keep `USE_REAL_GEE = False` for demo. Toggle True for real Sentinel-2 data.

In [3]:
USE_REAL_GEE = False  # 👈 Toggle True after: earthengine authenticate
DISTRICT     = 'Wardha'

if USE_REAL_GEE:
    import ee
    try:
        ee.Initialize()
        print('✅ GEE already authenticated!')
    except Exception:
        ee.Authenticate()
        ee.Initialize()
        print('✅ GEE authenticated!')
else:
    print('ℹ️  Synthetic mode — no GEE needed')
    print('   Stratified synthetic data mimics real Sentinel-2 distributions')

ℹ️  Synthetic mode — no GEE needed
   Stratified synthetic data mimics real Sentinel-2 distributions


## 📊 Cell 4 — Stratified Data Generation

### Why stratified instead of random?
```
Random sampling (old):          Stratified sampling (new):
SOC <5    :   8 samples ❌      SOC <5    : 200 samples ✅
SOC 5-10  : 142 samples         SOC 5-10  : 200 samples ✅
SOC 10-20 : 298 samples         SOC 10-20 : 200 samples ✅
SOC 20-35 :  47 samples ❌      SOC 20-35 : 200 samples ✅
SOC >35   :   5 samples ❌      SOC >35   : 200 samples ✅
```
Rare high-carbon fields are now equally represented!

In [4]:
from src.ingestion import generate_stratified_synthetic

N_PER_STRATUM = 200  # Samples per SOC stratum (200 × 5 = 1000 total)

if USE_REAL_GEE:
    from src.ingestion import (initialize_gee, get_study_area,
                               get_sentinel2_collection,
                               get_soilgrids_soc,
                               sample_pixels_stratified)
    initialize_gee()
    geometry  = get_study_area(DISTRICT)
    composite = get_sentinel2_collection(geometry)
    soc_layer = get_soilgrids_soc(geometry)
    raw_df    = sample_pixels_stratified(composite, soc_layer,
                                          geometry, n_samples=N_PER_STRATUM*5)
else:
    raw_df = generate_stratified_synthetic(
        n_per_stratum=N_PER_STRATUM, district=DISTRICT
    )

raw_df.to_csv('outputs/stratified_samples.csv', index=False)
print(f'\n💾 Saved: outputs/stratified_samples.csv')
print(f'📊 Total samples: {len(raw_df)}')
raw_df.head()

✅ Stratified synthetic data: 1000 samples (200 × 5 strata)
   Stratum 1 — Critical (<5 g/kg): 200 samples | SOC 1.0–5.0 g/kg
   Stratum 2 — Very Low (5-10 g/kg): 200 samples | SOC 5.0–10.0 g/kg
   Stratum 3 — Low-Moderate (10-20 g/kg): 200 samples | SOC 10.0–20.0 g/kg
   Stratum 4 — Good (20-35 g/kg): 200 samples | SOC 20.1–35.0 g/kg
   Stratum 5 — Excellent (>35 g/kg): 200 samples | SOC 35.0–54.8 g/kg

💾 Saved: outputs/stratified_samples.csv
📊 Total samples: 1000


,B2,B3,B4,B8,B11,B12,SOC,stratum,stratum_label,LandPractice,sampling_method,latitude,longitude
0,0.171363,0.168414,0.202628,0.189855,0.401286,0.221839,2.50,1,Critical (<5 g/kg),Conventional Tilled / Bare Soil,stratified_synthetic,20.471030,78.847626
1,0.120405,0.176081,0.256823,0.169190,0.410392,0.264854,4.80,1,Critical (<5 g/kg),Conventional Tilled / Bare Soil,stratified_synthetic,20.898525,78.527937
2,0.146032,0.234649,0.270949,0.184922,0.411523,0.333662,3.93,1,Critical (<5 g/kg),Conventional Tilled / Bare Soil,stratified_synthetic,20.465572,78.625787
3,0.141216,0.184399,0.316221,0.139310,0.458409,0.346228,3.39,1,Critical (<5 g/kg),Conventional Tilled / Bare Soil,stratified_synthetic,20.956368,78.726978
4,0.166152,0.209252,0.207334,0.172245,0.336154,0.224863,1.62,1,Critical (<5 g/kg),Conventional Tilled / Bare Soil,stratified_synthetic,20.851831,78.482833


## 📐 Cell 5 — Compute Spectral Indices
> NDVI (vegetation), BSI (bare soil), NDTI (tillage), NDWI (water), EVI, CRC (crop residue)

In [5]:
from src.indices import compute_indices_df

df = compute_indices_df(raw_df)

index_cols = ['NDVI','BSI','NDTI','NDWI','EVI','CRC']
print('📐 Spectral Index Statistics:')
print('='*55)
print(df[index_cols].describe().round(3).to_string())

✅ Computed 6 spectral indices. DataFrame shape: (1000, 19)
📐 Spectral Index Statistics:
           NDVI       BSI      NDTI      NDWI       EVI       CRC
count  1000.000  1000.000  1000.000  1000.000  1000.000  1000.000
mean      0.422    -0.115     0.113    -0.394     0.393     0.299
std       0.411     0.338     0.138     0.358     0.379     0.127
min      -0.498    -0.723    -0.224    -0.886    -0.312    -0.055
25%       0.097    -0.425     0.016    -0.721     0.070     0.207
50%       0.546    -0.095     0.111    -0.473     0.425     0.297
75%       0.791     0.177     0.202    -0.109     0.728     0.394
max       0.940     0.507     0.501     0.412     1.102     0.593


## 🌾 Cell 6 — Land Practice Classification

In [6]:
from src.indices import add_practice_labels

df = add_practice_labels(df)

print('🌾 Practice Distribution:')
counts = df['LandPractice'].value_counts()
total  = len(df)
for p, c in counts.items():
    pct = round(c/total*100,1)
    bar = '█' * int(pct/2)
    print(f'  {pct:5.1f}%  {bar:<25}  {p}')


🌾 Land Practice Distribution:
LandPractice
Cover Crop / Dense Vegetation      469
Conventional Tilled / Bare Soil    305
No-Till / Residue                  203
Mixed / Transitional                23
🌾 Practice Distribution:
   46.9%  ███████████████████████    Cover Crop / Dense Vegetation
   30.5%  ███████████████            Conventional Tilled / Bare Soil
   20.3%  ██████████                 No-Till / Residue
    2.3%  █                          Mixed / Transitional


## 🗺️ Cell 7 — Spatial Cross Validation Split

### Why spatial CV instead of random CV?
```
Normal CV (wrong):             Spatial CV (correct):
Train: pixel at lat=20.500     Train: entire North Wardha block
Test:  pixel at lat=20.501     Test:  entire South Wardha block
↑ Too close! Model 'cheats'    ↑ Truly independent!
↑ R² is FAKE high              ↑ R² is HONEST
```

In [7]:
from src.ingestion import spatial_cv_split

N_FOLDS = 5
df = spatial_cv_split(df, n_folds=N_FOLDS)

print('\n✅ Spatial fold assignment complete')
print('   Each fold = a different geographic block of Wardha')
print(f'   Training will rotate: train on {N_FOLDS-1} blocks, test on 1 block')

# Visualize fold distribution
print('\nFold sizes:')
for fold_id, group in df.groupby('spatial_fold'):
    lat_c = group['latitude'].mean()
    lon_c = group['longitude'].mean()
    print(f'  Fold {fold_id}: {len(group)} samples | '
          f'center ({lat_c:.2f}N, {lon_c:.2f}E)')


🗺️  Spatial CV: 5 geographic folds assigned
              count  mean_soc
spatial_fold                 
0               214     19.77
1               245     20.98
2               199     17.43
3               234     18.86
4               108     20.36

✅ Spatial fold assignment complete
   Each fold = a different geographic block of Wardha
   Training will rotate: train on 4 blocks, test on 1 block

Fold sizes:
  Fold 0: 214 samples | center (20.62N, 78.71E)
  Fold 1: 245 samples | center (20.76N, 78.59E)
  Fold 2: 199 samples | center (20.75N, 78.85E)
  Fold 3: 234 samples | center (20.86N, 78.73E)
  Fold 4: 108 samples | center (20.75N, 78.73E)


## 🤖 Cell 8 — Train Models with Spatial CV

In [8]:
from src.soc_model import SOCPredictor, PracticeClassifier
from src.ingestion import run_spatial_cross_validation

# ── SOC Model with Spatial CV ──────────────────────────────────
print('Step 1: Spatial Cross Validation (honest accuracy)...')
print('='*55)
soc_model = SOCPredictor(n_estimators=200)
cv_result = run_spatial_cross_validation(df, soc_model, n_folds=N_FOLDS)

# Final train on full dataset
print('\nStep 2: Final training on full stratified dataset...')
print('='*55)
metrics = soc_model.train(df)

# ── Practice Classifier ────────────────────────────────────────
print('\nStep 3: Practice Classifier...')
print('='*55)
prac_clf = PracticeClassifier()
clf_metrics = prac_clf.train(df)

print('\n' + '='*55)
print('📊 MODEL SUMMARY')
print('='*55)
print(f"  Normal CV R²    : {metrics['CV_R2_mean']} ± {metrics['CV_R2_std']}")
print(f"  Spatial CV R²   : {cv_result['r2_mean']} ± {cv_result['r2_std']}")
print(f"  RMSE            : {metrics['RMSE']} g/kg")
print(f"  Practice Acc    : {round(clf_metrics['accuracy']*100,1)}%")
print('\n💡 Note: Spatial CV R² is usually LOWER than normal CV')
print('   That is expected — it is the HONEST number to report in paper')

# Save models
soc_model.save('outputs/soc_model.pkl')
print('\n💾 Models saved!')

Step 1: Spatial Cross Validation (honest accuracy)...

🔬 Spatial 5-Fold Cross Validation
──────────────────────────────────────────────────
🌱 SOC MODEL TRAINED (with sample weights)
   R²        : 0.912
   RMSE      : 4.782 g/kg
   MAE       : 3.394 g/kg
   CV R²     : -6.437 ± 3.353
   Samples   : 786
──────────────────────────────────────────────────
  ✅ Fold 0: R²=0.916 | RMSE=4.47 g/kg | MAE=3.18 | n_test=214
──────────────────────────────────────────────────
🌱 SOC MODEL TRAINED (with sample weights)
   R²        : 0.897
   RMSE      : 4.727 g/kg
   MAE       : 3.263 g/kg
   CV R²     : -6.267 ± 5.721
   Samples   : 755
──────────────────────────────────────────────────
  ✅ Fold 1: R²=0.920 | RMSE=4.64 g/kg | MAE=3.35 | n_test=245
──────────────────────────────────────────────────
🌱 SOC MODEL TRAINED (with sample weights)
   R²        : 0.921
   RMSE      : 4.449 g/kg
   MAE       : 3.12 g/kg
   CV R²     : -3.011 ± 2.553
   Samples   : 801
─────────────────────────────────────────

## 📊 Cell 9 — Stratum-wise Accuracy
> Most important metric for research paper — shows model works across ALL carbon levels

In [9]:
print('🔬 Stratum-wise Accuracy Report')
print('(Is our model accurate for RARE high/low SOC fields?)')
print()

stratum_df = soc_model.stratum_accuracy(df)

print('\n📄 For research paper, report this table:')
print(stratum_df.to_string(index=False))
stratum_df.to_csv('outputs/stratum_accuracy.csv', index=False)
print('\n💾 Saved: outputs/stratum_accuracy.csv')

🔬 Stratum-wise Accuracy Report
(Is our model accurate for RARE high/low SOC fields?)


📊 Stratum-wise Accuracy (Research Metric):
────────────────────────────────────────────────────────────
  ⚠️  Critical (<5)          n=200 | R²=0.434 | RMSE=0.89 | MAE=0.73
  ⚠️  Very Low (5-10)        n=200 | R²=0.183 | RMSE=1.28 | MAE=1.02
  ⚠️  Low-Mod (10-20)        n=199 | R²=0.114 | RMSE=2.74 | MAE=2.14
  ⚠️  Good (20-35)           n=201 | R²=-0.020 | RMSE=4.35 | MAE=3.44
  ⚠️  Excellent (>35)        n=200 | R²=0.160 | RMSE=5.37 | MAE=4.01
────────────────────────────────────────────────────────────

📄 For research paper, report this table:
        Stratum   n     R2  RMSE   MAE
  Critical (<5) 200  0.434 0.885 0.726
Very Low (5-10) 200  0.183 1.276 1.017
Low-Mod (10-20) 199  0.114 2.737 2.142
   Good (20-35) 201 -0.020 4.349 3.435
Excellent (>35) 200  0.160 5.373 4.008

💾 Saved: outputs/stratum_accuracy.csv


## 📊 Cell 10 — Visualizations (LinkedIn Screenshots)

In [ ]:
# Run predictions on test sample
from src.indices import get_carbon_potential

test_raw = generate_stratified_synthetic(n_per_stratum=60, district=DISTRICT)
from src.indices import compute_indices_df, add_practice_labels
test_df  = compute_indices_df(test_raw)
test_df  = add_practice_labels(test_df)
test_df  = soc_model.predict(test_df)
test_df  = prac_clf.predict_proba_df(test_df)
test_df['CreditPotential_INR'] = test_df['PredictedPractice'].apply(
    lambda p: get_carbon_potential(p)['credit_potential_per_ha'])

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0e1117')
fig.suptitle('🌱 Soil Carbon Analysis — Wardha, Maharashtra',
             fontsize=15, color='white', fontweight='bold')

TEAL='#4ecdc4'; RED='#ff6b6b'; GOLD='#ffd93d'; GREEN='#6bcb77'
colors=[TEAL,GREEN,GOLD,RED,'#a78bfa','#f9844a']

# Plot 1: SOC Distribution
ax=axes[0,0]; ax.set_facecolor('#1a1a2e')
ax.hist(test_df['SOC_predicted'],bins=20,color=TEAL,edgecolor='#0e1117',alpha=0.85)
ax.axvline(20,color=RED,linestyle='--',lw=2,label='Healthy (20 g/kg)')
ax.axvline(test_df['SOC_predicted'].mean(),color=GOLD,linestyle=':',lw=2,
           label=f'Mean={test_df["SOC_predicted"].mean():.1f}')
ax.set_title('SOC Distribution',color='white'); ax.legend(fontsize=8)
ax.set_xlabel('SOC (g/kg)',color='#aaa'); ax.tick_params(colors='#888')

# Plot 2: Practice Pie
ax=axes[0,1]; ax.set_facecolor('#1a1a2e')
counts=test_df['PredictedPractice'].value_counts()
wedges,_,atext=ax.pie(counts.values,autopct='%1.1f%%',colors=colors[:len(counts)],
                       startangle=90,pctdistance=0.82,
                       wedgeprops={'edgecolor':'#0e1117','linewidth':2})
for t in atext: t.set_color('white'); t.set_fontsize(8)
ax.legend(wedges,counts.index,loc='lower center',bbox_to_anchor=(0.5,-0.3),
          fontsize=7,ncol=2,framealpha=0.1,labelcolor='white')
ax.set_title('Land Practices',color='white')

# Plot 3: SOC by Practice
ax=axes[0,2]; ax.set_facecolor('#1a1a2e')
pracs=test_df['PredictedPractice'].unique()
data=[test_df[test_df['PredictedPractice']==p]['SOC_predicted'].values for p in pracs]
bp=ax.boxplot(data,patch_artist=True,medianprops={'color':'white','linewidth':2})
for patch,c in zip(bp['boxes'],colors): patch.set_facecolor(c); patch.set_alpha(0.75)
ax.axhline(20,color=RED,linestyle='--',lw=1.5)
ax.set_title('SOC by Practice',color='white')
ax.set_xticks(range(1,len(pracs)+1))
ax.set_xticklabels([p[:15] for p in pracs],rotation=30,ha='right',fontsize=7,color='#aaa')
ax.tick_params(colors='#888')

# Plot 4: Stratum accuracy from Spatial CV
ax=axes[1,0]; ax.set_facecolor('#1a1a2e')
cv_folds=[f['fold'] for f in cv_result['fold_results']]
cv_r2s  =[f['r2']   for f in cv_result['fold_results']]
bars=ax.bar(cv_folds,cv_r2s,color=[GREEN if r>0.65 else RED for r in cv_r2s],
             edgecolor='#0e1117',alpha=0.85)
ax.axhline(cv_result['r2_mean'],color=GOLD,linestyle='--',lw=2,
           label=f'Mean={cv_result["r2_mean"]}')
for bar,r2 in zip(bars,cv_r2s):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
            f'{r2:.3f}',ha='center',color='white',fontsize=9)
ax.set_title('Spatial CV R² per Fold',color='white')
ax.set_xlabel('Geographic Fold',color='#aaa')
ax.set_ylabel('R²',color='#aaa'); ax.legend(fontsize=8)
ax.tick_params(colors='#888')

# Plot 5: SOC confidence intervals
ax=axes[1,1]; ax.set_facecolor('#1a1a2e')
sample=test_df.head(30).reset_index(drop=True)
ax.errorbar(sample.index,sample['SOC_predicted'],
            yerr=sample['SOC_std']*1.96,fmt='o',color=TEAL,
            ecolor='#555',capsize=3,markersize=4)
ax.axhline(20,color=RED,linestyle='--',lw=1.5,label='Healthy threshold')
ax.set_title('SOC Predictions + 95% CI',color='white')
ax.set_xlabel('Field Index',color='#aaa')
ax.set_ylabel('SOC (g/kg)',color='#aaa')
ax.legend(fontsize=8); ax.tick_params(colors='#888')

# Plot 6: Credit potential
ax=axes[1,2]; ax.set_facecolor('#1a1a2e')
cred=test_df.groupby('PredictedPractice')['CreditPotential_INR'].mean().sort_values()
hbars=ax.barh(cred.index,cred.values,color=colors[:len(cred)],alpha=0.85)
for bar,v in zip(hbars,cred.values):
    ax.text(v+50,bar.get_y()+bar.get_height()/2,
            f'₹{v:,.0f}',va='center',color='white',fontsize=8)
ax.set_title('Carbon Credit (₹/ha/yr)',color='white')
ax.set_yticklabels([p[:20] for p in cred.index],fontsize=7,color='#aaa')
ax.tick_params(colors='#888')

for ax in axes.flat:
    ax.spines['bottom'].set_color('#333')
    ax.spines['left'].set_color('#333')

plt.tight_layout()
plt.savefig('outputs/analysis_charts.png',dpi=150,
            bbox_inches='tight',facecolor='#0e1117')
plt.show()
print('✅ Charts saved: outputs/analysis_charts.png')

## 🗺️ Cell 11 — Interactive Folium Map
> Click any field dot to see SOC, practice, confidence and credit potential

In [ ]:
import folium

def soc_color(soc):
    if soc<5:    return '#d73027'
    elif soc<8:  return '#fc8d59'
    elif soc<12: return '#fee090'
    elif soc<16: return '#91cf60'
    elif soc<22: return '#1a9850'
    else:        return '#006837'

m = folium.Map(location=[test_df['latitude'].mean(),
                          test_df['longitude'].mean()],
               zoom_start=10, tiles='CartoDB dark_matter')

for _,row in test_df.iterrows():
    color = soc_color(row['SOC_predicted'])
    popup = f"""
    <div style='font-family:monospace;width:230px;font-size:12px;'>
    <b>📍 Field Report</b><hr>
    <b>SOC:</b>        {row['SOC_predicted']} g/kg<br>
    <b>95% CI:</b>     [{row['SOC_lower_95']} – {row['SOC_upper_95']}]<br>
    <b>Category:</b>   {row['SOC_category']}<br>
    <b>Practice:</b>   {row['PredictedPractice']}<br>
    <b>Confidence:</b> {row.get('PracticeConfidence','N/A')}%<br>
    <b>NDVI:</b>       {round(row.get('NDVI',0),3)}<br>
    <b>BSI:</b>        {round(row.get('BSI',0),3)}<br><hr>
    <b>💰 Credit:</b>  ₹{row['CreditPotential_INR']:,}/ha/yr
    </div>"""
    folium.CircleMarker(
        location=[row['latitude'],row['longitude']],
        radius=max(5,row.get('PracticeConfidence',70)/15),
        color=color,fill=True,fill_color=color,fill_opacity=0.75,
        popup=folium.Popup(popup,max_width=260),
        tooltip=f"SOC:{row['SOC_predicted']} g/kg | {row['PredictedPractice'][:18]}"
    ).add_to(m)

legend='''
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;
background:rgba(0,0,0,0.85);padding:14px;border-radius:10px;
font-family:monospace;color:white;font-size:12px;border:1px solid #444;">
<b>🌱 SOC (g/kg)</b><br><br>
<span style="color:#d73027">●</span> &lt;5 Critical<br>
<span style="color:#fc8d59">●</span> 5–8 Very Low<br>
<span style="color:#fee090">●</span> 8–12 Low<br>
<span style="color:#91cf60">●</span> 12–16 Moderate<br>
<span style="color:#1a9850">●</span> 16–22 Good<br>
<span style="color:#006837">●</span> &gt;22 Excellent
</div>'''
m.get_root().html.add_child(folium.Element(legend))
m.save('outputs/soc_map.html')
print('✅ Map saved: outputs/soc_map.html')
m

## 🤖 Cell 12 — Run Agentic Pipeline

In [ ]:
import sys
sys.path.insert(0, PROJECT_ROOT)

# Import from correct location
try:
    from src.agent import run_mock_agent
except ModuleNotFoundError:
    # Fallback: agent.py might be in classifier/ folder
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'classifier'))
    from agent import run_mock_agent

field_data = test_df.to_dict('records')

print('🚀 Launching 5-node Soil Carbon Agent...')
result = run_mock_agent(field_data, region_name=f'{DISTRICT}, Maharashtra')

print('\n' + '='*55)
node_names = ['🔍 PERCEPTION','📊 ANALYSIS','🩺 DIAGNOSIS',
              '💡 RECOMMENDATIONS','📋 REPORT']
for i, step in enumerate(result['reasoning_trace']):
    print(f'\n── Node {i+1}: {node_names[i]} ──')
    if i == 3:
        for rec in result['recommendations']:
            print(f"  [{rec['priority']}] {rec['practice']}")
            print(f"         {rec['carbon_benefit']} | ₹{rec['credit_potential_inr_per_ha']:,}/ha")
    else:
        content = step.split(':',1)[-1].strip() if ':' in step else step
        print(content[:500])

## 📋 Cell 13 — Print Report + Save All Outputs

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result['final_report']))

# Save all outputs
with open('outputs/soil_carbon_report.md','w',encoding='utf-8') as f:
    f.write(result['final_report'])

pd.DataFrame(result['recommendations']).to_csv('outputs/recommendations.csv',index=False)
test_df.to_csv('outputs/field_predictions.csv',index=False)
pd.DataFrame(cv_result['fold_results']).to_csv('outputs/spatial_cv_results.csv',index=False)
stratum_df.to_csv('outputs/stratum_accuracy.csv',index=False)

print('💾 All outputs saved:')
outputs = [
    '📄 outputs/soil_carbon_report.md',
    '📊 outputs/field_predictions.csv',
    '💡 outputs/recommendations.csv',
    '🔬 outputs/spatial_cv_results.csv',
    '📈 outputs/stratum_accuracy.csv',
    '🗺️  outputs/soc_map.html',
    '📉 outputs/analysis_charts.png',
]
for o in outputs: print(f'  {o}')
print('\n🎉 Pipeline complete — ready for LinkedIn + GitHub!')

## 🚀 Cell 14 — Launch Streamlit Dashboard
> Run this OR open a terminal and run: `streamlit run dashboard/app.py`

In [ ]:
import subprocess, threading, time, webbrowser

dashboard_path = os.path.join(PROJECT_ROOT, 'dashboard', 'app.py')

if not os.path.exists(dashboard_path):
    print(f'❌ Not found: {dashboard_path}')
    print('   Run manually: streamlit run dashboard/app.py')
else:
    print('🚀 Launching dashboard at http://localhost:8501')
    print('   To stop: press ■ (interrupt) button above')
    threading.Thread(
        target=lambda: (time.sleep(3), webbrowser.open('http://localhost:8501')),
        daemon=True
    ).start()
    subprocess.run([sys.executable,'-m','streamlit','run',dashboard_path])

---
## ✅ Complete!

### What upgraded in this version:
| Feature | Old | New |
|---------|-----|-----|
| Sampling | Random | Stratified (5 strata) |
| CV Method | Normal K-Fold | Spatial K-Fold |
| Sample weights | None | Inverse-frequency |
| Accuracy report | Overall only | Per-stratum |
| Confidence intervals | None | 95% CI from RF trees |

### For your research paper:
- Report **Spatial CV R²** (not normal CV)
- Include **stratum-wise accuracy** table
- Cite: Stratified sampling methodology
- Validate on real LUCAS soil dataset in Phase 2

---
*🌱 Soil Carbon Agentic AI | Research-Grade v2 | Sentinel-2 · LangGraph · Streamlit*